# Data Preprocessing and Time-Series Preparation

## Palm Oil Waste Prediction Using Stacking Ensemble Machine Learning

This notebook prepares the palm oil dataset for machine learning while preserving its chronological structure.

The preprocessing workflow includes:

- Loading and verifying the dataset
- Defining predictor and target variables
- Creating chronological training and independent test datasets
- Preparing rolling-origin validation splits
- Applying model-specific preprocessing where required
- Preparing sequential data for LSTM modelling

The independent test period is kept separate from model development and is not used for hyperparameter tuning or model selection.

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

## 1. Load and Verify the Dataset

The processed monthly dataset is loaded and checked before defining the modelling variables.

In [2]:
df = pd.read_csv("../data/palm_oil_waste_dataset_1964_2024.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (732, 14)


,Year,Month,Month_Number,Quarter,FFB_1000MT,EFB_1000MT,PKS_1000MT,POME_1000MT,Mesocarp_Fibre_1000MT,Avg_Temperature_C,Rainfall_mm,Humidity_pct,Wind_Speed_ms,Solar_Radiation_Wm2
0,1964,Jan,1,Q1,26.85,5.907,1.611,17.453,3.491,27.27,45.84,82.20,2.00,187.66
1,1964,Feb,2,Q1,32.22,7.088,1.933,20.943,4.189,28.28,67.60,81.26,2.11,192.43
2,1964,Mar,3,Q1,53.70,11.814,3.222,34.905,6.981,28.26,154.10,84.20,2.20,181.76
3,1964,Apr,4,Q2,69.81,15.358,4.189,45.377,9.075,28.22,221.79,87.13,2.09,167.18
4,1964,May,5,Q2,75.18,16.540,4.511,48.867,9.773,27.24,308.64,88.19,2.00,157.86


## 2. Define Predictor and Target Variables

The modelling dataset contains temporal, production, and environmental predictors. Four palm oil waste streams are treated as target variables.

FFB is retained as a predictor to reproduce the primary modelling framework used in the MSc study. However, because the target waste quantities were derived from FFB using conversion factors, model performance with FFB should be interpreted alongside sensitivity analysis excluding this variable.

In [3]:
features = [
    "Year",
    "Month_Number",
    "Quarter",
    "FFB_1000MT",
    "Avg_Temperature_C",
    "Rainfall_mm",
    "Humidity_pct",
    "Wind_Speed_ms",
    "Solar_Radiation_Wm2"
]

targets = [
    "EFB_1000MT",
    "PKS_1000MT",
    "POME_1000MT",
    "Mesocarp_Fibre_1000MT"
]

print("Predictor variables:")
for feature in features:
    print("-", feature)

print("\nTarget variables:")
for target in targets:
    print("-", target)

Predictor variables:
- Year
- Month_Number
- Quarter
- FFB_1000MT
- Avg_Temperature_C
- Rainfall_mm
- Humidity_pct
- Wind_Speed_ms
- Solar_Radiation_Wm2

Target variables:
- EFB_1000MT
- PKS_1000MT
- POME_1000MT
- Mesocarp_Fibre_1000MT


In [4]:
print("Quarter values:", df["Quarter"].unique())
print("Quarter data type:", df["Quarter"].dtype)

Quarter values: <StringArray>
['Q1', 'Q2', 'Q3', 'Q4']
Length: 4, dtype: str
Quarter data type: str


### 2.1 Encoding the Quarter Variable

The `Quarter` variable is stored as categorical text (Q1–Q4). It is converted to numerical values (1–4) to make it compatible with the machine learning models.

Month number is retained alongside quarter because it provides finer monthly temporal information, while quarter provides a broader seasonal grouping.

In [5]:
quarter_mapping = {
    "Q1": 1,
    "Q2": 2,
    "Q3": 3,
    "Q4": 4
}

df["Quarter"] = df["Quarter"].map(quarter_mapping)

df[["Month", "Month_Number", "Quarter"]].head(12)

,Month,Month_Number,Quarter
0,Jan,1,1
1,Feb,2,1
2,Mar,3,1
3,Apr,4,2
4,May,5,2
5,Jun,6,2
6,Jul,7,3
7,Aug,8,3
8,Sep,9,3
9,Oct,10,4


### 2.2 Create Feature and Target Matrices

The predictor variables are separated from the four target waste variables to create the feature matrix (`X`) and target matrix (`y`).

In [6]:
X = df[features].copy()
y = df[targets].copy()

print("Feature matrix shape:", X.shape)
print("Target matrix shape:", y.shape)

X.head()

Feature matrix shape: (732, 9)
Target matrix shape: (732, 4)


,Year,Month_Number,Quarter,FFB_1000MT,Avg_Temperature_C,Rainfall_mm,Humidity_pct,Wind_Speed_ms,Solar_Radiation_Wm2
0,1964,1,1,26.85,27.27,45.84,82.20,2.00,187.66
1,1964,2,1,32.22,28.28,67.60,81.26,2.11,192.43
2,1964,3,1,53.70,28.26,154.10,84.20,2.20,181.76
3,1964,4,2,69.81,28.22,221.79,87.13,2.09,167.18
4,1964,5,2,75.18,27.24,308.64,88.19,2.00,157.86


## 3. Chronological Training and Independent Test Split

Because the dataset represents a time series, the observations are divided chronologically rather than randomly.

- **Model development period:** January 1964 – December 2021
- **Independent test period:** January 2022 – December 2024

The 2022–2024 observations are reserved as an independent test set and are not used during model training, hyperparameter tuning, or model selection. This provides a more realistic assessment of model performance on future unseen observations.

In [7]:
train_mask = df["Year"] <= 2021
test_mask = df["Year"] >= 2022

X_train = X.loc[train_mask].copy()
X_test = X.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_test = y.loc[test_mask].copy()

print("Training features:", X_train.shape)
print("Training targets:", y_train.shape)
print("Test features:", X_test.shape)
print("Test targets:", y_test.shape)

Training features: (696, 9)
Training targets: (696, 4)
Test features: (36, 9)
Test targets: (36, 4)


In [8]:
print(
    "Training period:",
    df.loc[train_mask, ["Year", "Month"]].iloc[0].to_dict(),
    "to",
    df.loc[train_mask, ["Year", "Month"]].iloc[-1].to_dict()
)

print(
    "Test period:",
    df.loc[test_mask, ["Year", "Month"]].iloc[0].to_dict(),
    "to",
    df.loc[test_mask, ["Year", "Month"]].iloc[-1].to_dict()
)

Training period: {'Year': 1964, 'Month': 'Jan'} to {'Year': 2021, 'Month': 'Dec'}
Test period: {'Year': 2022, 'Month': 'Jan'} to {'Year': 2024, 'Month': 'Dec'}


## 4. Rolling-Origin Time-Series Validation

Model development uses rolling-origin validation rather than random cross-validation.

Ten validation folds are created. Each fold expands the training window chronologically and uses the following year as the validation period.

The validation years range from 2012 to 2021. Earlier observations are always used to predict later observations, preserving the temporal structure of the dataset.

The independent 2022–2024 test set remains completely excluded from this process.

In [9]:
validation_years = range(2012, 2022)

rolling_splits = []

for val_year in validation_years:
    
    train_indices = df.index[
        (df["Year"] >= 1964) &
        (df["Year"] < val_year)
    ].to_numpy()
    
    val_indices = df.index[
        df["Year"] == val_year
    ].to_numpy()
    
    rolling_splits.append((train_indices, val_indices))

print("Number of validation splits:", len(rolling_splits))

Number of validation splits: 10


In [10]:
for split_number, (train_idx, val_idx) in enumerate(rolling_splits, start=1):
    
    train_start_year = df.loc[train_idx, "Year"].min()
    train_end_year = df.loc[train_idx, "Year"].max()
    val_year = df.loc[val_idx, "Year"].unique()[0]
    
    print(
        f"Split {split_number:02d}: "
        f"Train {train_start_year}-{train_end_year} "
        f"({len(train_idx)} months) | "
        f"Validate {val_year} ({len(val_idx)} months)"
    )

Split 01: Train 1964-2011 (576 months) | Validate 2012 (12 months)
Split 02: Train 1964-2012 (588 months) | Validate 2013 (12 months)
Split 03: Train 1964-2013 (600 months) | Validate 2014 (12 months)
Split 04: Train 1964-2014 (612 months) | Validate 2015 (12 months)
Split 05: Train 1964-2015 (624 months) | Validate 2016 (12 months)
Split 06: Train 1964-2016 (636 months) | Validate 2017 (12 months)
Split 07: Train 1964-2017 (648 months) | Validate 2018 (12 months)
Split 08: Train 1964-2018 (660 months) | Validate 2019 (12 months)
Split 09: Train 1964-2019 (672 months) | Validate 2020 (12 months)
Split 10: Train 1964-2020 (684 months) | Validate 2021 (12 months)


## 5. Model-Specific Feature Scaling

Feature scaling is applied according to the requirements of each model.

- **Random Forest:** original feature values are retained.
- **XGBoost:** original feature values are retained.
- **LSTM:** features are standardised because neural networks are sensitive to differences in variable scale.
- **Elastic Net:** stacking inputs are standardised during meta-model development.

To prevent data leakage, scaling parameters are estimated only from the relevant training data and then applied to validation or test observations.

In [11]:
# Demonstration using the first rolling-origin split

train_idx, val_idx = rolling_splits[0]

X_fold_train = X.loc[train_idx].copy()
X_fold_val = X.loc[val_idx].copy()

scaler = StandardScaler()

X_fold_train_scaled = scaler.fit_transform(X_fold_train)
X_fold_val_scaled = scaler.transform(X_fold_val)

print("Fold training shape:", X_fold_train_scaled.shape)
print("Fold validation shape:", X_fold_val_scaled.shape)

print(
    "\nTraining years:",
    df.loc[train_idx, "Year"].min(),
    "to",
    df.loc[train_idx, "Year"].max()
)

print(
    "Validation year:",
    df.loc[val_idx, "Year"].unique()[0]
)

Fold training shape: (576, 9)
Fold validation shape: (12, 9)

Training years: 1964 to 2011
Validation year: 2012


## 6. LSTM Sequence Preparation

Unlike the tree-based models, the LSTM processes sequences of consecutive observations.

A sequence length of 12 months is used so that each sample contains one full year of monthly information. Each sequence therefore has the structure:

`12 time steps × 9 predictor variables`

The target associated with each sequence corresponds to the final month of that sequence.

In [12]:
def create_lstm_sequences(X_data, y_data, sequence_length=12):
    X_sequences = []
    y_sequences = []
    sequence_indices = []

    for i in range(sequence_length - 1, len(X_data)):
        start = i - sequence_length + 1
        end = i + 1

        X_sequences.append(X_data[start:end])
        y_sequences.append(y_data[i])
        sequence_indices.append(i)

    return (
        np.array(X_sequences),
        np.array(y_sequences),
        np.array(sequence_indices)
    )

In [13]:
y_fold_train = y.loc[train_idx].to_numpy()

X_seq, y_seq, seq_idx = create_lstm_sequences(
    X_fold_train_scaled,
    y_fold_train,
    sequence_length=12
)

print("Original feature shape:", X_fold_train_scaled.shape)
print("LSTM feature shape:", X_seq.shape)

print("Original target shape:", y_fold_train.shape)
print("LSTM target shape:", y_seq.shape)

print("Sequence index shape:", seq_idx.shape)

Original feature shape: (576, 9)
LSTM feature shape: (565, 12, 9)
Original target shape: (576, 4)
LSTM target shape: (565, 4)
Sequence index shape: (565,)


### 6.1 Preparing Validation Sequences

Validation sequences require historical context from the end of the training period.

For each validation year, the preceding 11 months of predictor data are combined with the validation-year predictors. This allows the LSTM to construct a 12-month sequence for every validation month while ensuring that validation target values are not used as model inputs.

For example, the sequence used to predict January 2012 contains predictor information from February 2011 through January 2012.

In [14]:
sequence_length = 12

# First rolling-origin split: training through 2011, validation on 2012
train_idx, val_idx = rolling_splits[0]

X_fold_train = X.loc[train_idx].copy()
X_fold_val = X.loc[val_idx].copy()

y_fold_train = y.loc[train_idx].copy()
y_fold_val = y.loc[val_idx].copy()

# Fit scaler using training data only
lstm_scaler = StandardScaler()

X_fold_train_scaled = lstm_scaler.fit_transform(X_fold_train)
X_fold_val_scaled = lstm_scaler.transform(X_fold_val)

# Add the final 11 training months as historical context
X_val_with_context = np.vstack([
    X_fold_train_scaled[-(sequence_length - 1):],
    X_fold_val_scaled
])

print("Historical context months:",
      sequence_length - 1)

print("Validation months:",
      len(X_fold_val_scaled))

print("Combined validation input shape:",
      X_val_with_context.shape)

Historical context months: 11
Validation months: 12
Combined validation input shape: (23, 9)


In [15]:
X_val_sequences = []

for i in range(len(X_fold_val_scaled)):
    start = i
    end = i + sequence_length

    X_val_sequences.append(
        X_val_with_context[start:end]
    )

X_val_sequences = np.array(X_val_sequences)

y_val_sequences = y_fold_val.to_numpy()

print("Validation LSTM features:",
      X_val_sequences.shape)

print("Validation LSTM targets:",
      y_val_sequences.shape)

Validation LSTM features: (12, 12, 9)
Validation LSTM targets: (12, 4)


### 6.2 Verify Sequence Alignment

The validation sequences are checked against their calendar dates to confirm that each 12-month input window ends in the month associated with its target value.

In [16]:
# Dates corresponding to the training and validation observations
train_dates = pd.to_datetime(
    dict(
        year=df.loc[train_idx, "Year"],
        month=df.loc[train_idx, "Month_Number"],
        day=1
    )
).reset_index(drop=True)

val_dates = pd.to_datetime(
    dict(
        year=df.loc[val_idx, "Year"],
        month=df.loc[val_idx, "Month_Number"],
        day=1
    )
).reset_index(drop=True)

# Final 11 training dates + all validation dates
context_dates = pd.concat([
    train_dates.iloc[-(sequence_length - 1):],
    val_dates
]).reset_index(drop=True)

print(
    "First validation sequence:",
    context_dates.iloc[0].strftime("%b %Y"),
    "to",
    context_dates.iloc[sequence_length - 1].strftime("%b %Y")
)

print(
    "Last validation sequence:",
    context_dates.iloc[-sequence_length].strftime("%b %Y"),
    "to",
    context_dates.iloc[-1].strftime("%b %Y")
)

First validation sequence: Feb 2011 to Jan 2012
Last validation sequence: Jan 2012 to Dec 2012


### 6.3 Reusable LSTM Fold Preparation

A reusable function is created to prepare each rolling-origin fold for LSTM modelling. For every fold, the scaler is fitted only on the historical training observations. The validation data are transformed using the training-fitted scaler, and the final 11 training months are retained as historical context for constructing validation sequences.

In [17]:
def prepare_lstm_fold(X, y, train_idx, val_idx, sequence_length=12):
    
    # Select training and validation observations
    X_train_fold = X.loc[train_idx].copy()
    X_val_fold = X.loc[val_idx].copy()

    y_train_fold = y.loc[train_idx].copy()
    y_val_fold = y.loc[val_idx].copy()

    # Fit scaler on training data only
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train_fold)
    X_val_scaled = scaler.transform(X_val_fold)

    # Create training sequences
    X_train_seq, y_train_seq, _ = create_lstm_sequences(
        X_train_scaled,
        y_train_fold.to_numpy(),
        sequence_length
    )

    # Add historical context to validation predictors
    X_val_context = np.vstack([
        X_train_scaled[-(sequence_length - 1):],
        X_val_scaled
    ])

    # Create one sequence for each validation month
    X_val_seq = []

    for i in range(len(X_val_scaled)):
        X_val_seq.append(
            X_val_context[i:i + sequence_length]
        )

    X_val_seq = np.array(X_val_seq)
    y_val_seq = y_val_fold.to_numpy()

    return (
        X_train_seq,
        y_train_seq,
        X_val_seq,
        y_val_seq,
        scaler
    )

In [18]:
for split_number, (train_idx, val_idx) in enumerate(
    rolling_splits,
    start=1
):
    
    X_train_seq, y_train_seq, X_val_seq, y_val_seq, scaler = (
        prepare_lstm_fold(
            X,
            y,
            train_idx,
            val_idx,
            sequence_length=12
        )
    )

    print(
        f"Split {split_number:02d}: "
        f"Train {X_train_seq.shape} | "
        f"Validation {X_val_seq.shape}"
    )

Split 01: Train (565, 12, 9) | Validation (12, 12, 9)
Split 02: Train (577, 12, 9) | Validation (12, 12, 9)
Split 03: Train (589, 12, 9) | Validation (12, 12, 9)
Split 04: Train (601, 12, 9) | Validation (12, 12, 9)
Split 05: Train (613, 12, 9) | Validation (12, 12, 9)
Split 06: Train (625, 12, 9) | Validation (12, 12, 9)
Split 07: Train (637, 12, 9) | Validation (12, 12, 9)
Split 08: Train (649, 12, 9) | Validation (12, 12, 9)
Split 09: Train (661, 12, 9) | Validation (12, 12, 9)
Split 10: Train (673, 12, 9) | Validation (12, 12, 9)


## 7. Final LSTM Training and Independent Test Preparation

After model development and hyperparameter selection, the final LSTM model can be trained using the complete 1964–2021 development period.

The feature scaler is fitted exclusively on the 1964–2021 data and then applied to the independent 2022–2024 test predictors.

As with rolling-origin validation, the final 11 months of the training period are retained as historical context. This enables a prediction sequence to be constructed for January 2022 without using any test-period target values.

In [19]:
sequence_length = 12

# Fit scaler on the complete development period only
final_lstm_scaler = StandardScaler()

X_train_scaled = final_lstm_scaler.fit_transform(X_train)
X_test_scaled = final_lstm_scaler.transform(X_test)

# Create final training sequences
X_train_lstm, y_train_lstm, _ = create_lstm_sequences(
    X_train_scaled,
    y_train.to_numpy(),
    sequence_length=sequence_length
)

# Add final 11 training months as context for the test period
X_test_context = np.vstack([
    X_train_scaled[-(sequence_length - 1):],
    X_test_scaled
])

# Construct one sequence for every test month
X_test_lstm = []

for i in range(len(X_test_scaled)):
    X_test_lstm.append(
        X_test_context[i:i + sequence_length]
    )

X_test_lstm = np.array(X_test_lstm)
y_test_lstm = y_test.to_numpy()

print("Final LSTM training features:", X_train_lstm.shape)
print("Final LSTM training targets:", y_train_lstm.shape)

print("Independent test features:", X_test_lstm.shape)
print("Independent test targets:", y_test_lstm.shape)

Final LSTM training features: (685, 12, 9)
Final LSTM training targets: (685, 4)
Independent test features: (36, 12, 9)
Independent test targets: (36, 4)


In [20]:
all_dates = pd.to_datetime(
    dict(
        year=df["Year"],
        month=df["Month_Number"],
        day=1
    )
)

train_dates = all_dates.loc[train_mask].reset_index(drop=True)
test_dates = all_dates.loc[test_mask].reset_index(drop=True)

test_context_dates = pd.concat([
    train_dates.iloc[-(sequence_length - 1):],
    test_dates
]).reset_index(drop=True)

print(
    "First test sequence:",
    test_context_dates.iloc[0].strftime("%b %Y"),
    "to",
    test_context_dates.iloc[sequence_length - 1].strftime("%b %Y")
)

print(
    "Last test sequence:",
    test_context_dates.iloc[-sequence_length].strftime("%b %Y"),
    "to",
    test_context_dates.iloc[-1].strftime("%b %Y")
)

First test sequence: Feb 2021 to Jan 2022
Last test sequence: Jan 2024 to Dec 2024


## 8. Sensitivity Analysis Feature Set Without FFB

Exploratory analysis showed that the four waste targets have near-deterministic relationships with FFB because the waste quantities were derived using approximately fixed conversion factors.

To assess the extent to which model performance depends on FFB, an alternative feature set is created with `FFB_1000MT` excluded.

This feature set will later be used for sensitivity analysis and compared with the primary models that include FFB.

In [21]:
features_without_ffb = [
    feature for feature in features
    if feature != "FFB_1000MT"
]

X_without_ffb = df[features_without_ffb].copy()

X_train_without_ffb = X_without_ffb.loc[train_mask].copy()
X_test_without_ffb = X_without_ffb.loc[test_mask].copy()

print("Features with FFB:", len(features))
print("Features without FFB:", len(features_without_ffb))

print("\nTraining shape without FFB:",
      X_train_without_ffb.shape)

print("Test shape without FFB:",
      X_test_without_ffb.shape)

print("\nFeatures used:")
for feature in features_without_ffb:
    print("-", feature)

Features with FFB: 9
Features without FFB: 8

Training shape without FFB: (696, 8)
Test shape without FFB: (36, 8)

Features used:
- Year
- Month_Number
- Quarter
- Avg_Temperature_C
- Rainfall_mm
- Humidity_pct
- Wind_Speed_ms
- Solar_Radiation_Wm2


## 9. Conversion-Factor Baseline

Because the target waste quantities are closely related to FFB through approximately fixed conversion factors, a simple deterministic baseline is defined for later comparison with the machine learning models.

This baseline provides an important reference point for determining whether the machine learning models offer predictive improvement beyond the known FFB-to-waste relationships.

In [22]:
conversion_factors = {
    "EFB_1000MT": 0.22,
    "PKS_1000MT": 0.06,
    "POME_1000MT": 0.65,
    "Mesocarp_Fibre_1000MT": 0.13
}

conversion_factors

{'EFB_1000MT': 0.22,
 'PKS_1000MT': 0.06,
 'POME_1000MT': 0.65,
 'Mesocarp_Fibre_1000MT': 0.13}

## 10. Preprocessing Summary

The dataset has been prepared for machine learning while preserving its chronological structure.

Key preprocessing decisions include:

- Nine predictor variables and four palm oil waste targets were defined.
- The categorical quarter variable was converted to numerical form.
- Data from 1964–2021 were reserved for model development, while 2022–2024 were isolated as an independent test period.
- Ten expanding-window rolling-origin validation folds were created for model development.
- Scaling for LSTM modelling is fitted exclusively on the relevant training period to prevent information leakage.
- Twelve-month LSTM sequences were constructed while retaining historical context across training-validation and training-test boundaries.
- An alternative eight-feature dataset excluding FFB was created for sensitivity analysis.
- A simple FFB conversion-factor baseline was defined for comparison with the machine learning models.

These preprocessing steps provide the foundation for reproducible model development and evaluation.